# Read the raw events from the .hdf5 files
This example script loads raw events. First, load the required libraries.

In [1]:
import h5py
import hdf5plugin
import pandas as pd
from event_reader.eventslicer import EventSlicer

Then input the path to the .hdf5 file you want to load.

In [ ]:
# load events
event_file = "" # path to the dataset/LeftEvent/LeftEvent.hdf5 or dataset/RightEvent/RightEvent.hdf5
event_loader = EventSlicer(h5py.File(event_file, 'r'))

Then set the start timestamp and end timestamp of the event slice you want to get.

In [ ]:
start_time = # the global start timestamp of a desired event slice
end_time = # the global end timestamp of a desired event slice
assert start_time < end_time, f"Make sure the 'end_time={end_time}' is larger than the 'start_time={start_time}'."
assert start_time >= event_loader.get_start_time_us()*1e-6, f"The input 'start_time={start_time}' is smaller than the start timestamp of the event file '{event_loader.get_start_time_us()*1e-6}'"
assert end_time <= event_loader.get_final_time_us()*1e-6, f"The input 'end_time={end_time}' is larger than the final timestamp of the event file '{event_loader.get_final_time_us()*1e-6}'"

<span style="color: red;">Attention:</span> Several recording sequences contain empty or invalid data, which should not be loaded. We provide statistics for the invalid time periods in [`defect_data_stats.csv`](defect_data_stats.csv). We can check whether an input time period overlaps an invalid period as follows:

In [ ]:
# load invalid time periods
defect_stats = pd.read_csv('defect_data_stats.csv')
def check_data(event_file, start_time, end_time):
    for i, s in defect_stats["Session_ID"].items():
        a = defect_stats.loc[i, "Activity"]
        if s in event_file and a in event_file:
            # The loaded event file contains invalid periods
            # next is to check whether the input timestamps are in the defect period
            if end_time < defect_stats.loc[i, "start_t"] or start_time > defect_stats.loc[i, "end_t"]:
                continue
            else:
                raise NotImplementedError(
                    f"The input time period [{start_time}, {end_time}] overlaps with the invalid time period [{defect_stats.loc[i, 'start_t']}, {defect_stats.loc[i, 'end_t']}]. Please avoid loading data from this period."
                )
check_data(event_file, start_time, end_time)

After the check we can load the event slice as follows.

In [ ]:
# need to convert the unit of time to microsecond to call get_events()
start_time *= 1e6
end_time *= 1e6
event_slice = event_loader.get_events(start_time, end_time)

The loaded event slice is now stored in ***event_slice***. ***EventSlicer*** provides multiple functions for accessing HDF5 files; please refer to [`eventslicer.py`](event_reader/eventslicer.py).